In [ ]:
from jablonski import (
    Simulator,
    SingletState,
    SpectroscopicSystem,
    TripletState,
    Parameter,
    initial,
    assign,
)
from jablonski.transitions import (
    Absorption,
    EnergyTransferUpconversion,
    Fluorescence
)
from jablonski.plots import jablonski_diagram, model_report
from poincare import Simulator, SteadyState

import pint 

import numpy as np
import matplotlib.pyplot as plt

u = pint.get_application_registry()

In [ ]:
class ThreeStateSystem(SpectroscopicSystem):
    ground_poblation = 1e0

    pump: Parameter = assign(default=0 / u.cm**2 / u.s)

    GSA_rate: Parameter = assign()
    ESA_rate: Parameter = assign()
    ETU_rate: Parameter = assign()

    N1_decay_rate: Parameter = assign()
    N2_decay_rate: Parameter = assign()

    N0: SingletState = initial(
        energy=0 * u.eV, spin_multiplicity="singlet", default=ground_poblation
    )

    N1: SingletState = initial(energy=1 * u.eV, spin_multiplicity="singlet", default=0)

    N2: SingletState = initial(energy=2 * u.eV, spin_multiplicity="singlet", default=0)

    ESA01 = Absorption(ground=N0, excited=N1, rate=GSA_rate, pump=pump)

    ESA12 = Absorption(ground=N1, excited=N2, rate=ESA_rate , pump=pump)

    ETU12 = EnergyTransferUpconversion(
        sensitizer=N1, activator=N1, relaxator=N2, rate=ETU_rate
    )

    # Radiative decays in the order of ~ 10^3
    DECAY10 = Fluorescence(ground=N0, excited=N1, rate= N1_decay_rate)

    DECAY20 = Fluorescence(ground=N0, excited=N2, rate=N2_decay_rate)


In [ ]:
system = ThreeStateSystem(
    GSA_rate=10e-21 * u.cm**2,
    ESA_rate=10e-20 * u.cm**2,
    ETU_rate=0 / u.s,
    N1_decay_rate=100 / u.s,
    N2_decay_rate=1000 / u.s,
    pump=0 /u.s/u.cm**2
)
sim = Simulator(system)
steady = SteadyState()

res = steady.sweep(
    sim, variable=ThreeStateSystem.pump, values=np.logspace(-3, 4, 50) / u.s / u.cm**2
)

In [ ]:
plt.figure()
plt.plot(res.pump, res.N0, label ="N0" )
plt.plot(res.pump, res.N1, label ="N1" )
plt.plot(res.pump, res.N2, label ="N2" )

plt.legend()
plt.savefig("here")

plt.xscale('log')
plt.yscale('log')

In [ ]:
tex = model_report(ThreeStateSystem, standalone=True, replace_algebraics=True)
with open('report.tex', 'w') as f:
    f.write(tex)